# 🧭 GIADA roadmap Task 10 — sufficienza dell'ingresso
Confronto appaiato delle informazioni sul percorso di voltaggio e dei sottopassi della formula Ca-HVA. Questa è la Task 10 originale; `TG-01` è uno studio supplementare distinto. Tutte le viste con futuro teacher sono oracle diagnostici, non input del neurone autonomo.


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_roadmap_task10');GIADA_REPO=WORK/'giada';TEACHER_REPO=WORK/'neuron_as_deep_net'
assert not WORK.exists(),'Usa una sessione Kaggle nuova: directory già presente.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip();print({'revision':REVISION})


## 📁 Input
Servono il dataset `hayflow-targeted-transition-dataset-v1-1-base` completo di HDF5/schema/manifest e il piccolo artefatto `giada_physiological_voltage_paths.zip` della Task 9. Non servono lo ZIP della Task 7, Task 9b/9c o `TG-01`. Non serve GPU né compilare NEURON: usiamo la formula `.mod` e percorsi già registrati.


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula
from src.giada_teacher.physiological_path_floor import load_verified_task9
from src.giada_teacher.roadmap_input_sufficiency import run_roadmap_input_sufficiency,VIEWS,SUBSTEPS
INPUT_ROOT=Path('/kaggle/input');override=os.environ.get('GIADA_TASK9_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
 candidates+=list(INPUT_ROOT.rglob('giada_physiological_voltage_paths.zip'))
 candidates += [p for p in INPUT_ROOT.rglob('archive.zip') if 'physiological' in str(p).lower()]
 candidates += [p.parent for p in INPUT_ROOT.rglob('selected_paths.json') if (p.parent/'final_report.json').is_file()]
TASK9_SOURCE=None
for path in candidates:
 if not path.exists():continue
 try:load_verified_task9(path);TASK9_SOURCE=path.resolve();break
 except (RuntimeError,FileNotFoundError,ValueError,KeyError):continue
assert TASK9_SOURCE is not None,'Artefatto Task 9 esatto non trovato.'
base_override=os.environ.get('GIADA_TARGETED_DATASET')
base_candidates=[Path(base_override).expanduser()] if base_override else []
base_candidates += [Path('/kaggle/input/datasets/alessandrobelli/hayflow-targeted-transition-dataset-v1-1-base')]
if INPUT_ROOT.is_dir():base_candidates += [p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]
BASE_ROOT=next((p.resolve() for p in base_candidates if p.is_dir() and all((p/name).is_file() for name in ('transition_dataset.h5','state_schema.json','dataset_manifest.json'))),None)
assert BASE_ROOT is not None,'Dataset targeted base esatto non trovato: servono HDF5, schema e manifest.'
display({'task9_source':str(TASK9_SOURCE),'dataset':str(BASE_ROOT),'views':VIEWS,'substeps':SUBSTEPS})


In [ ]:
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_roadmap_task10_input_sufficiency')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Usa una sessione nuova.'
def progress(percent,label):print(f'[GIADA roadmap Task 10][SHA-256 {label}] {percent}%',flush=True)
report=run_roadmap_input_sufficiency(formula,BASE_ROOT,TASK9_SOURCE,OUTPUT_DIR,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'path_count':report['task9_selected_path_count'],'midpoint_reproduction':report['full41_midpoint_reproduction_max_abs'],'teacher_floor':report['full_path_teacher_floor'],'minimum_diagnostic_oracle_view':report['minimum_view_at_registered_0p001_gate'],'causal_views':report['causally_available_views'],'sealed_test_opened':report['sealed_test_opened']})
assert report['valid'] and report['full41_midpoint_reproduction_max_abs']<=1e-12 and not report['sealed_test_opened']


In [ ]:
import pandas as pd
rows=[]
for group,data in report['groups'].items():
 for view in ('start_only','endpoints_oracle','endpoints_mean_oracle','five_oracle','nine_oracle','twentyone_oracle'):
  m=data['views'][view]['40']['vs_full_path']
  rows.append({'group':group,'view':view,'m_excess':round(m['m_rmse'],6),'h_excess':round(m['h_rmse'],6),'open_excess':round(m['open_rmse'],6)})
display(pd.DataFrame(rows).pivot(index='group',columns='view',values='m_excess'))
print('Solo start_only è disponibile causalmente a t: tutte le altre colonne sono oracle teacher-forced.')


## 📦 Scarica il report
Metodo Blob/base64 concordato; non includiamo l'HDF5 nel file ZIP.


In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task10_input_sufficiency','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
